In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pymatgen.core import Element
from copy import deepcopy
import seaborn as sns
import os
from progressbar import ProgressBar

pd.set_option("display.max_rows", 1000)


In [ ]:

def get_data():
    """データ取得

    Returns:
        pd.DataFrame: データ。
        [str]: 元素名リスト。
        [str]: 目的変数名リスト。
        [str]: メタカラム名リスト。
        int: 目的へs縫うの分割数。
    """
    import json
    filepath = os.path.join("../data_calculated/hea4_phys_condition.json")
    with open(filepath, "r") as f:
        cond = json.load(f)
    ndiv = cond["NDIV"] # digitizeする分割数。

    # 加工済みデータの読み込み
    element_labels = []
    for i in range(4):
        element_labels.append("element{}".format(i+1))
    target_names = ['M', 'TC', 'R', ]
    meta_names = ['heakey', ]
    filepath = os.path.join("../data_calculated/hea4_phys.csv")
    dfraw = pd.read_csv(filepath)
    return dfraw, element_labels, target_names, meta_names, ndiv


g_dfraw, g_element_labels, g_target_names, g_meta_names, g_ndiv = get_data()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
def apply_regression(df, descriptor_names, target_name):
    """apply fit and predict 
    
    Args:
        df (pd.DataFrame): data.
        descriptor_names ([str]): 説明変数カラム名リスト。
        target_naem (str): 目的変数カラム名。
    """
    X = df[descriptor_names].values
    y = df[target_name].values
    X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=0)
    reg = RandomForestRegressor()
    reg.fit(X_train, y_train)
    yp_train = reg.predict(X_train)
    yp_test = reg.predict(X_test)
    r2_train = r2_score(y_train, yp_train)
    r2_test = r2_score(y_test,yp_test)
    print("R2",r2_train, r2_test)
    
    if True:
        ylim = (y.min(), y.max())
        fig, ax = plt.subplots(figsize=(5,5))
        ax.scatter(y_test,yp_test, s=1, alpha=0.1)
        ax.set_xlim(ylim)
        ax.set_ylim(ylim)
        ax.plot(ylim,ylim, "--", c="red")
    
apply_regression(g_dfraw, ['group_mean', 'group_std', 'row_mean', 'row_std'], "R")

## itemset miningでの再解析

回帰を行った問題をitemset miningでも解析してみます。
この際に、説明変数生成時に除いた情報（構成元素）を含めることが可能です。


#### 頻出マイニングのmoduleのimport

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

### 頻出マイニング

In [ ]:
# グラフ図示関数を定義しておく
from typing import List


def show_rules(df: pd.DataFrame, filename=None, figsize=(5, 5)):
    """ruleの図示。

    Args:
        df (pd.DataFrame): データ。
        figsize (tuple, optional): 図のサイズ. Defaults to (5, 5).
    """
    df = df.copy()
    # antecedentsとconsequentsはfrozen setという形式で入っている。
    # frozen setだと図示した時に見にくいのでフォーマットの変更を行う。
    df["antecedents"] = df["antecedents"].apply(lambda x: next(iter(x)))
    df["consequents"] = df["consequents"].apply(lambda x: next(iter(x)))

    import networkx as nx
    import matplotlib.pyplot as plt
    plt.figure(figsize=figsize)
    GA = nx.from_pandas_edgelist(df,
                                 source='antecedents', target='consequents',
                                 create_using=nx.MultiDiGraph())
    nx.draw(GA, node_color="yellow", edge_color="lightblue",
            arrowsize=20, connectionstyle="arc3,rad=0.1",
            font_color="red", with_labels=True)
    # plt.tight_layout() はincompatibleだと言われるのでmarginで制御する。
    plt.margins(0.3)
    if filename is not None:
        plt.savefig(filename)
    plt.show()


In [ ]:
def convert_to_transaction(df, element_labels,
                           feature_id_labels=["M_id", "TC_id", "R_id",
                                              "group_mean_id", "group_std_id",
                                              "row_mean_id", "row_std_id",
                                              'n_group3', 'n_group4', 'n_group5', 'n_group6', 'n_group7',
                                              'n_group8', 'n_group9', 'n_group10', 'n_group11', 'n_group12',
                                              'n_group13', 'n_group14', 'n_group15',
                                              ]):
    """transactionへの変換。

    Args:
        df (pd.DataFrame): data.
        element_labels ([str])): 元素名リスト
        feature_id_labels ([str]), optional): itemとして使用するカラム名リスト. Defaults to ["M_id", "TC_id", "R_id", "group_mean_id", "group_std_id", "row_mean_id", "row_std_id", 'n_group3', 'n_group4', 'n_group5', 'n_group6', 'n_group7', 'n_group8', 'n_group9', 'n_group10', 'n_group11', 'n_group12', 'n_group13', 'n_group14', 'n_group15', ].

    Returns:
        list: transaction
    """

    discretevalues = {}

    for idname in element_labels:
        discretevalues[idname] = df[idname].values.tolist()

    for idname in feature_id_labels:
        value_list = []
        for value in df[idname].values.tolist():
            if idname == "R_id":
                valueid = "{}=={}".format(idname, value)
            elif idname in ["M_id", "TC_id"]:
                if value > 1:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""  # ignore M and TC
            elif idname.startswith("group") or idname.startswith("row"):
                if value > 0:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""
            elif idname.startswith("n_"):
                if value > 1:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""
            else:
                valueid = "{}=={}".format(idname, value)
            value_list.append(valueid)
        discretevalues[idname] = value_list

    df_discrete = pd.DataFrame(discretevalues)

    transaction = []
    for values_raw in df_discrete.values:
        values_raw = values_raw.tolist()
        values = list(filter(None, values_raw))  # listから””を除く。
        transaction.append(values)

    return transaction


In [ ]:
def make_rule(dfraw, element_labels, ndiv, min_support=0.3, min_threshold=0.8):
    """rule miningを行う。

    Args:
        dfraw (pd.DataFrame): データ。
        element_labels ([str])): itemとして変換する要素。
        ndiv (int): 要素がintやfloatの型の場合のitemの分割数。
        min_support (float, optional): supportの最小値. Defaults to 0.3.
        min_threshold (float, optional): thresholdの最小値. Defaults to 0.8.

    Returns:
        pd.DataFrame: データ。
    """
    df_rules_list = []
    for i in range(ndiv):
        target_condition = "R_id=={}".format(i+1)
        # 「多数」に結果が引きずられるので、制限する。
        # 制限しないとsupportも見るのでR_idの最も大きなpeakの情報が主として出てくる。
        dfq = dfraw.query(target_condition).reset_index(drop=True)
        transaction = convert_to_transaction(dfq, element_labels,)

        te = TransactionEncoder()
        te.fit(transaction)
        te_ary = te.fit(transaction).transform(transaction)
        df = pd.DataFrame(te_ary, columns=te.columns_)
        df_freq_items = apriori(
            df, min_support=min_support, max_len=10000, use_colnames=True, verbose=1)
        df_freq_items.sort_values(by="support", ascending=False)
        df_rules = association_rules(df_freq_items, metric="confidence",
                                     min_threshold=min_threshold)
        df_rules = df_rules.sort_values(
            by="support", ascending=False).reset_index(drop=True)
        df_rules_list.append(df_rules, )
        if not os.path.isdir("image_executed"):
            os.makedirs("image_executed")
        show_rules(df_rules, filename=os.path.join("image_executed/target_{}.png".format(i)), figsize=(5, 5))
    return df_rules_list


g_df_rules_list = make_rule(g_dfraw, g_element_labels, g_ndiv)


一つの図にまとめます。
![回帰のitemset miningの図のまとめ](image_keep/hea4_phys_condition_targetid.cyjs.trim.png)


図でも示しましたが、回帰は全体をみます。
そして、回帰で見えるのはgroup_std_id vs R_idです。

頻出マイニングでもR_idを変えた全ての振る舞いを見ると
期待通りにR_id $\sin$ group_std_idとなる
ことが分かります。

また、説明変数に変換する際は元素の並びがあると困るので、説明変数に変換する際に捨てた元素そのものは用いませんでした。
頻出マイニングはそのような変数も同時に扱えます。これによりR_idが大きい場合には回帰では捨てた元素の特徴が出てくることが分かりました。

最後のdf_rulesの表示をします。

In [ ]:
g_df_rules_list[-1]

supportの再計算を行い数値の確認をしておきます。

In [ ]:
def make_support(dfraw, sentense, elm):
    """元データからsupportの計算を行う。

    Args:
        dfraw (pd.DataFrame): データ。
        sentense (str): query文。
        elm (str): elementsカラムに含まれる元素名。
    """
    dfq = dfraw.query(sentense)
    print(dfq.shape)
    dfq2 = dfq[dfq["elements"].str.contains(",{},".format(elm))]
    print(dfq2.shape)
    print("support=", dfq2.shape[0]/dfq.shape[0])


make_support(g_dfraw, "R_id==10", "Sc")


上図にR_idが大きい場合に元素名が現れました。
現れた元素の一つであるScがあるとR_idの分布が変わるのでしょうか。

分布の可視化を行う。


In [ ]:
# 全元素名を得る。
def get_all_elm(dfraw):
    """全元素を得る。

    Args:
        dfraw (pd.DataFrame):データ。

    Returns:
        np.ndarray: unique元素名リスト
    """
    elm1 = dfraw["element1"].values
    elm2 = dfraw["element2"].values
    elm3 = dfraw["element3"].values
    elm4 = dfraw["element4"].values
    uique_elms = np.unique(np.hstack([elm1, elm2, elm3, elm4]))
    return uique_elms


g_uique_elms = get_all_elm(g_dfraw)
print(g_uique_elms)


２つの分布があったとして、それらの（仮想的な）母集団が同じかどうかを
t-valueで評価する。

ref.
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html

Calculate the T-test for the means of two independent samples of scores.
```python
scipy.stats.ttest_ind(a, b)
```

なお、母集団を考えなくてもサンプルの分布の偏り具合を示す指標として用いることができます。

In [ ]:
import collections


def plot_selected_hist(df, elements, ndiv, label="R_id", filename=None):
    """elementsを含むlabelカラムのhistogramを示す。

    Args:
        df (pd.DataFrame): データ。
        elements ([str]): 元素名リスト。
        ndiv (int): labelカラムの値の分割数。
        label (str, optional): 物理量のカラム. Defaults to "R_id".
    """
    dfselect = df.copy()
    for elm in elements:
        dfselect = dfselect[dfselect["elements"].str.contains(elm)]

    # Rのhistogramを生成して規格化
    Rall = df[label].values
    counter = collections.Counter(Rall)
    hist_Rall = []
    for i in range(ndiv):
        hist_Rall.append(counter[i])
    n_sum = np.sum(hist_Rall)
    hist_Rall = hist_Rall/n_sum

    # dfselect["R"]のhistogramを生成して規格化
    Rselect = dfselect[label].values
    counter = collections.Counter(Rselect)
    hist_Rselect = []
    for i in range(ndiv):
        hist_Rselect.append(counter[i])
    n_sum = np.sum(hist_Rselect)
    hist_Rselect = hist_Rselect/n_sum

    # 可視化
    fig, ax = plt.subplots()
    width = 1
    center = list(range(1, ndiv+1))
    ax.bar(center, hist_Rall, width=width,
           alpha=0.5,  align='center', label="all")
    ax.bar(center, hist_Rselect, width=width, alpha=0.5,
           align='center', label=str(elements))
    ax.set_xlabel(label)
    ax.set_ylabel("normalized occurrence")
    ax.legend()
    if filename is not None:
        fig.savefig(filename)
        print("save to", filename)


g_filename = "image_executed/{}.png".format("_".join(["Sc", "In"]))
plot_selected_hist(g_dfraw, ["Sc", "In"], g_ndiv,
                  filename=g_filename
                  )
# 分布が偏っている。


上の図はR_idのoccurence1１を全体で１に規格化した分布を表しています。

In [ ]:
from scipy.stats import ttest_ind
from itertools import combinations
import random


def calc_tpvalues(df,  elms, ncombi=1, name="R", ):
    """t-valueの計算

    Args:
        df (pd.DataFrame): データ。
        elms ([str]): 元素名リスト。
        ncombi (int, optional): 元素組み合わせ数. Defaults to 1.
        name (str, optional): 対象カラム名. Defaults to "R".

    Returns:
        pd.DataFrame: データ。
    """
    Rall = df[name].values
    tplist = []
    comblist = list(combinations(elms, ncombi))


    for i, elm2 in enumerate(comblist):

        dfs = df
        for elm1 in elm2:
            dfs = dfs[dfs["elements"].str.contains(","+elm1+",")]
        Rselect = dfs[name].values

        if True:
            # 今の場合は母集団は分かっているが、
            # ランダム化法で母集団からランダムに取る。
            # 全部使っても同じ。
            population = list(range(Rall.shape[0]))
            id_ = np.array(random.sample(population, Rselect.shape[0]))
            Rrand = df.loc[id_, name]
        else:
            Rrand = Rall

        t, p = ttest_ind(Rrand, Rselect)
        tplist.append([elm2, t, p])
    df_tp = pd.DataFrame(tplist, columns=["elmements", "tvalue", "pvalue"])
    df_tp.sort_values(by="tvalue")
    return df_tp.sort_values(by="tvalue").reset_index(drop=True)


g_df_tp1 = calc_tpvalues(g_dfraw,  g_uique_elms, ncombi=1)


In [ ]:
display(g_df_tp1.head())
display(g_df_tp1.tail())


（当然ですが）t-valueで分布がどちら側にずれているのか分かります。

In [ ]:
plot_selected_hist(g_dfraw, ["Sc"], g_ndiv, )
plot_selected_hist(g_dfraw, ["Rh"], g_ndiv,)


In [ ]:
g_df_tp2 = calc_tpvalues(g_dfraw,  g_uique_elms, ncombi=2)


In [ ]:
display(g_df_tp2.head(15))
display(g_df_tp2.tail())


上位と下位の比較を行う。

In [ ]:
plot_selected_hist(g_dfraw, ["In", "Sc"], g_ndiv)
plot_selected_hist(g_dfraw, ["Ge", "Si"], g_ndiv)


教師なし学習で妥当な予測モデルが作成できない場合は、分布の違いを議論しても良いでしょう。

### 補足
#### group_std vs Rの可視化

In [ ]:
g_dfraw.plot.scatter(x="group_std", y="R", s=1)
g_dfraw.plot.scatter(x="group_mean", y="R", s=1)
g_dfraw.plot.scatter(x="group_std", y="group_mean", s=1)


group_stdが小さいほどRが小さいことが分かります。
元素を４つ組み合わせるので、
group_stdとgroup_meanは独立ではなく、
group_stdが大きい場合は、group_meanも大きくなるという関係があります。